# Poking at the matches

This is the little notebook I use to look at the output of `pipeline_all.py`. Nothing fancy — load the latest matches CSV, see what paired up, and eyeball where the prices disagree.

Run the pipeline first (`python pipeline_all.py`) so there's something in `historical_data/similar_events/`.

In [ ]:
import glob, os
import pandas as pd

files = sorted(glob.glob("../historical_data/similar_events/similar_events_sbert_*.csv"))
assert files, "No matches found — run `python pipeline_all.py` first."

df = pd.read_csv(files[-1], low_memory=False)
print(f"Loaded {len(df):,} matched pairs from {os.path.basename(files[-1])}")
df.head()

## Strongest matches

Anything near 1.0 is basically the same question on both platforms.

In [ ]:
cols = ["similarity_score", "kalshi_title_raw", "poly_title_raw"]
df.sort_values("similarity_score", ascending=False)[cols].head(15)

## Tighten the threshold

The pipeline keeps everything above 0.70. Bump it up if you only want the high-confidence pairs.

In [ ]:
THRESHOLD = 0.90
strong = df[df["similarity_score"] >= THRESHOLD]
print(f"{len(strong):,} pairs at >= {THRESHOLD} similarity")
strong[cols].head(20)

## Where do the prices disagree?

Rough first pass only — this ignores fees, slippage, the bid/ask spread, and how each platform settles. It just lines up the Kalshi implied probability against the Polymarket YES price for pairs where both have a quote. Treat it as a list of things to look into, not a signal.

In [ ]:
g = df.copy()
g = g[(g["similarity_score"] >= 0.90)]
g["kalshi_mid_price"] = pd.to_numeric(g["kalshi_mid_price"], errors="coerce")
g["poly_yes_price"] = pd.to_numeric(g["poly_yes_price"], errors="coerce")
g = g.dropna(subset=["kalshi_mid_price", "poly_yes_price"])

g["kalshi_prob"] = g["kalshi_mid_price"] / 100.0   # Kalshi mid is in cents
g["price_gap"] = (g["kalshi_prob"] - g["poly_yes_price"]).abs()

out = ["kalshi_title_raw", "kalshi_prob", "poly_yes_price", "price_gap", "similarity_score"]
g.sort_values("price_gap", ascending=False)[out].head(20)